In [1]:
import sys 
!"{sys.executable}" -m pip install networkx numpy scipy torch scikit-learn matplotlib
# torch-geometric may require special install steps on Windows and is optional for this notebook



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

# Set random seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [3]:
import os
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

# Create empty graph
G = nx.Graph()

# Load C_elegans edgelist as an unweighted graph
path = os.path.join("..", "Datasets", "US_airports.txt")

# Each line: V1 V2 weight. We ignore the weight and use binary edges.
edges = np.loadtxt(path, dtype=int, usecols=(0, 1))

# If the file has a single edge, np.loadtxt returns a 1D array, so normalize it.
if edges.ndim == 1:
    edges = edges.reshape(1, 2)

G.add_edges_from(edges)

# Basic info
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Draw graph only if it is reasonably small; otherwise skip plotting.
if G.number_of_nodes() <= 200:
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, with_labels=True, node_size=50, font_size=8)
    plt.show()
else:
    print("Graph is large; skipping full plot.")

Nodes: 500
Edges: 2980
Graph is large; skipping full plot.


In [4]:
# adjacency matrix
nodelist = list(G.nodes())
A = nx.to_numpy_array(G, nodelist=nodelist)
print(A)

[[0. 1. 1. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 [1. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


In [5]:
#Degree
deg = np.array([G.degree(n) for n in nodelist])
deg

array([145,  15,  40,   1,   4,  35,  34,  30,   6,   5,  33,  26,   8,
        53,  68,   1,   7,  28,  70,  15,   8,   5,   8,  15,  66,  94,
        34,  16,   4,  12,   2, 114,   6,  14,  22,  64, 122, 132,   1,
        19, 114,  13,  18,   7,  85,   7,   2,  46,   2,   9,   9,   2,
         5,  16,  18,  18,   4,  30,  21,  11,   3,  75, 109,  12,   2,
        40,  15,   7,  11,  30,  44,  74,  76,  14,  67,  18,   3,   4,
        67,   2,  72,  16,  61,   4,  84,   4,  20,  42,  53,   5,  10,
         7,   5, 130,  40,  13,  35,  22,  21,  21, 136,  24,   6,  24,
        46,   6,   7,  89,  91,   8, 110,  11,  15,  27,  15,  42,  18,
        10,  22,  30,  42,  29,  12,   8,  30,  75,  62,   6,  34,  67,
        29,  26,  11,  98,   5,  23,  12,   7,  59,   6,  19,  17,  17,
         2,   7,   9,   3,   8,   5,  24,  14,  11,   4,   6,   3,   5,
         4,   4,  14,   8,   6,  28,   3,   7,   9,   5,   6,   4,   5,
         7,  16,   6,   1,  17,   3,   9,   5,   5,   2,   1,   

In [6]:
dist = dict(nx.all_pairs_shortest_path_length(G))
dist

{np.int64(0): {np.int64(0): 0,
  np.int64(110): 1,
  np.int64(48): 1,
  np.int64(282): 1,
  np.int64(176): 1,
  np.int64(71): 1,
  np.int64(55): 1,
  np.int64(47): 1,
  np.int64(136): 1,
  np.int64(151): 1,
  np.int64(50): 1,
  np.int64(67): 1,
  np.int64(155): 1,
  np.int64(40): 1,
  np.int64(19): 1,
  np.int64(302): 1,
  np.int64(122): 1,
  np.int64(61): 1,
  np.int64(21): 1,
  np.int64(100): 1,
  np.int64(118): 1,
  np.int64(138): 1,
  np.int64(157): 1,
  np.int64(89): 1,
  np.int64(37): 1,
  np.int64(11): 1,
  np.int64(46): 1,
  np.int64(81): 1,
  np.int64(120): 1,
  np.int64(128): 1,
  np.int64(206): 1,
  np.int64(17): 1,
  np.int64(172): 1,
  np.int64(45): 1,
  np.int64(78): 1,
  np.int64(25): 1,
  np.int64(5): 1,
  np.int64(2): 1,
  np.int64(235): 1,
  np.int64(85): 1,
  np.int64(7): 1,
  np.int64(181): 1,
  np.int64(63): 1,
  np.int64(149): 1,
  np.int64(16): 1,
  np.int64(143): 1,
  np.int64(183): 1,
  np.int64(30): 1,
  np.int64(225): 1,
  np.int64(126): 1,
  np.int64(127): 1

In [7]:
n = len(nodelist)
dist_matrix = np.zeros((n, n))

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and v in dist[u]:
            dist_matrix[i, j] = dist[u][v]

print(dist_matrix)

[[0. 1. 1. ... 3. 4. 3.]
 [1. 0. 2. ... 4. 5. 4.]
 [1. 2. 0. ... 3. 5. 4.]
 ...
 [3. 4. 3. ... 0. 6. 5.]
 [4. 5. 5. ... 6. 0. 4.]
 [3. 4. 4. ... 5. 4. 0.]]


4. Node Feature Extraction (Exact Formulas)

Paper defines three measures.

NLI  = Local influence

NGI  = Global influence

NLGC = Hybrid influence

### 4.1 Global Influence (NGI)

**Paper formula:**

$$\text{NGI}_i = \sum_{i \neq j} \frac{\sqrt{d(v_j) + \alpha}}{d_{ij}}$$

**Where:**
*   $d(v_j)$ = degree of node $j$
*   $d_{ij}$ = shortest path distance between node $i$ and node $j$
*   $\alpha$ = constant parameter

### 🔹 Node Global Influence (NGI) – Description

**Definition:**
NGI is a metric used to measure the overall importance of a node by considering its interaction with all other nodes in the network.

**Core Idea:**
It combines both:
*   **Local information** → node degree
*   **Global information** → shortest path distance

**Computation:**
For each node, influence is calculated by summing contributions from all other nodes based on:
*   **Smoothed degree** of the contributing node (using square root scaling)
*   **Distance** between the two nodes

**Role of Degree ($d(v_j)$):**
Nodes with higher connections contribute more influence. However, instead of using the raw degree directly, a square root transformation is applied to moderate the dominance of high-degree nodes.

**Role of Distance ($d_{ij}$):**
Influence decreases as distance increases, ensuring closer nodes have stronger impact. The inverse relationship gives higher weight to nearby nodes.

**Role of $\alpha$ (alpha):**
*   Added inside the square root to stabilize the computation.
*   Prevents zero or very small degree values from reducing influence too much.
*   Helps in smoothing the contribution of nodes.
*   $\alpha = 0.5$ provides a balanced contribution.

**Key Advantage:**
NGI captures both local connectivity and global positioning while ensuring balanced influence using square root scaling.

**Interpretation:**
A node with a higher NGI value is more influential in the network, considering both its connectivity and its position relative to other nodes.

In [8]:
alpha = 0.5

NGI = np.zeros(n)

for i, u in enumerate(nodelist):
    for j, v in enumerate(nodelist):
        if i != j and dist_matrix[i, j] != 0:
            NGI[i] += np.sqrt(deg[j] + alpha) / dist_matrix[i, j]

print("NGI:", NGI)

NGI: [1032.74839765  671.54151648  775.73578716  585.07962085  594.36757627
  747.02920486  773.03689077  765.33497776  615.99659898  632.87257968
  786.51867839  745.21678716  650.06605974  860.34448818  885.80755453
  585.07962085  646.87153159  755.05805608  897.59531266  691.93089802
  644.0716321   626.26194692  633.40100255  690.52638674  881.19574685
  920.31462222  801.33545319  714.90772662  616.50992478  675.03641824
  590.90139811  980.08669622  614.75455867  628.65012576  736.58361993
  874.79993533  974.42129034 1008.9303275   585.07962085  734.90052398
  985.69145578  707.51369159  698.83708479  647.53617205  947.06639392
  605.32991221  590.90139811  829.66039916  590.90139811  649.38003515
  664.38643399  590.90139811  616.29438846  707.47171152  705.96340783
  703.14713288  595.62589882  724.28326152  706.40432916  662.14440039
  607.11286393  905.33590306  974.21434484  699.90233422  590.90139811
  816.56155142  652.64130907  669.17078864  654.94824327  754.54036825
 

### 4.2 Node Local Influence (NLI)

**Formula:**

$$NLI_i = \frac{d(v_i) \times \log_2 \left( \sum_{j \in N_i} e^{d(v_j)} \right)}{n}$$

---

### 🔹 Node Local Influence (NLI) – Description

**Definition:**
NLI is a metric used to measure the importance of a node based on its local neighborhood structure, focusing only on its immediate connections.

**Core Idea:**
It captures influence using:
*   **Node’s own degree**
*   **Contribution from its neighboring nodes**

**Computation:**
For each node, influence is calculated by:
1. Taking the degree of the node.
2. Multiplying it with the logarithm of the sum of exponential contributions from its neighbors.
3. Normalizing by the total number of nodes ($n$).

**Role of Degree ($d(v_i)$):**
The degree reflects direct connectivity; nodes with more neighbors have higher local influence.

**Role of Neighbor Contribution:**
Each neighbor contributes via an exponential function, which:
*   Amplifies the importance of highly connected neighbors.
*   Highlights strong local structures.

**Role of Logarithm ($\\log$):**
*   Compresses large values from exponential growth.
*   Prevents numerical explosion and ensures balanced scaling.

**Role of Normalization ($n$):**
Dividing by total nodes ensures values are comparable across different graph sizes.

**Key Advantage:**
NLI focuses purely on local structure, making it effective in identifying nodes that are well-connected locally and surrounded by influential neighbors.

**Interpretation:**
A node with higher NLI is more influential within its immediate neighborhood, even if it is not globally central.

In [9]:
NLI = np.zeros(n)

for i, u in enumerate(nodelist):

    neighbors = list(G.neighbors(u))

    # Sum of exponential terms (using neighbor count here as influence proxy)
    exp_sum = 0
    for v in neighbors:
        exp_sum += np.exp(G.degree(v))   # you can modify this part if Ne_i(v_i) defined differently

    if exp_sum > 0:
        NLI[i] = (G.degree(u) * np.log2(exp_sum)) / n
    else:
        NLI[i] = 0

print("NLI:", NLI)

#calculation Verified

NLI: [5.69085035e+01 6.27572877e+00 1.67352770e+01 4.18381562e-01
 1.67352625e+00 1.46433672e+01 1.42249855e+01 1.25514578e+01
 2.51028937e+00 2.09190959e+00 1.38066035e+01 1.08779300e+01
 3.34705535e+00 2.21742420e+01 2.84499709e+01 4.18381562e-01
 2.92867347e+00 1.17146939e+01 2.92867348e+01 6.27572887e+00
 3.34705534e+00 2.09190959e+00 3.34705249e+00 6.27572887e+00
 2.76132071e+01 3.93279010e+01 1.42249855e+01 6.69411081e+00
 1.67352627e+00 5.02058302e+00 8.36763124e-01 4.76955395e+01
 2.51028937e+00 5.85734187e+00 9.20440236e+00 2.67764432e+01
 5.10425949e+01 5.52264133e+01 4.18381562e-01 7.94925658e+00
 4.76955395e+01 5.43896503e+00 7.53087464e+00 2.92867343e+00
 3.55624637e+01 2.92867093e+00 8.36763124e-01 1.92455686e+01
 8.36763124e-01 3.76543406e+00 3.76543732e+00 8.36763124e-01
 2.09190781e+00 6.69411081e+00 7.53087464e+00 7.53087464e+00
 1.67352625e+00 1.25514471e+01 8.78602029e+00 4.60220117e+00
 1.25514469e+00 3.13786444e+01 4.56036299e+01 5.02058310e+00
 8.36763124e-01 1.6

4.3 Hybrid Influence

Paper multiplies them.

$$\text{NLGC}_i = \text{NLI}_i \times \text{NGI}_i$$

In [10]:
NLGC = NLI * NGI
NLGC

array([5.87721658e+04, 4.21441241e+03, 1.29821533e+04, 2.44786526e+02,
       9.94689740e+02, 1.09390229e+04, 1.09964385e+04, 9.60606965e+03,
       1.54632972e+03, 1.32391222e+03, 1.08591516e+04, 8.10641607e+03,
       2.17580708e+03, 1.90774869e+04, 2.52011992e+04, 2.44786526e+02,
       1.89447549e+03, 8.84527401e+03, 2.62876359e+04, 4.34237071e+03,
       2.15574340e+03, 1.31008337e+03, 2.12002641e+03, 4.33355638e+03,
       2.43326406e+04, 3.61940423e+04, 1.13989852e+04, 4.78567154e+03,
       1.03174556e+03, 3.38907638e+03, 4.94444500e+02, 4.67457637e+04,
       1.54321183e+03, 3.68221870e+03, 6.77981201e+03, 2.34240308e+04,
       4.97369912e+04, 5.57196032e+04, 2.44786526e+02, 5.84191283e+03,
       4.70130858e+04, 3.84814223e+03, 5.26285448e+03, 1.89642198e+03,
       3.36800142e+04, 1.77281212e+03, 4.94444500e+02, 1.59672861e+04,
       4.94444500e+02, 2.44519771e+03, 2.50170547e+03, 4.94444500e+02,
       1.28923104e+03, 4.73589403e+03, 5.31652192e+03, 5.29531291e+03,
      

### 🔹 Multi-Scale Feature Construction

**Definition:**
Multi-scale feature construction is used to capture node influence at different neighborhood levels by progressively aggregating information from neighboring nodes.

**Core Idea:**
Instead of relying on a single-scale measure, influence is computed across multiple levels:
*   **Level 1** → node itself
*   **Level 2** → node + immediate neighbors
*   **Level 3** → node + extended neighborhood

---

### 🧬 Computation: NLI-based Features

**Level 1:**
$$W_{NLI1}(i) = NLI_i$$

**Level 2:**
$$W_{NLI2}(i) = W_{NLI1}(i) + \sum_{j \in N(i)} W_{NLI1}(j)$$

**Level 3:**
$$W_{NLI3}(i) = W_{NLI2}(i) + \sum_{j \in N(i)} W_{NLI2}(j)$$

---

### 🌍 Computation: NGI-based Features

**Level 1:**
$$W_{NGI1}(i) = NGI_i$$

**Level 2:**
$$W_{NGI2}(i) = W_{NGI1}(i) + \sum_{j \in N(i)} W_{NGI1}(j)$$

**Level 3:**
$$W_{NGI3}(i) = W_{NGI2}(i) + \sum_{j \in N(i)} W_{NGI2}(j)$$

---

### ✅ Key Advantages
*   **Higher-Order Influence:** Captures both local and extended neighborhood importance.
*   **Rich Structural Info:** Provides a multidimensional view of a node's position for learning.
*   **Propagation Awareness:** Helps GCNs understand how influence spreads across multiple hops.

**Interpretation:**
Nodes with higher values at deeper levels (e.g., $NLI_3$, $NGI_3$) are not only locally important but are also strategically connected to other influential regions in the graph.

In [11]:
import numpy as np

nodelist = list(G.nodes())
node_index = {node: i for i, node in enumerate(nodelist)}
n = len(nodelist)

# --- NLI Multi-scale ---
W_NLI1 = NLI.copy()
W_NLI2 = np.zeros(n)
W_NLI3 = np.zeros(n)

# NLI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI1[j]
    W_NLI2[i] = W_NLI1[i] + neighbor_sum

# NLI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NLI2[j]
    W_NLI3[i] = W_NLI2[i] + neighbor_sum


# --- NGI Multi-scale ---
W_NGI1 = NGI.copy()
W_NGI2 = np.zeros(n)
W_NGI3 = np.zeros(n)

# NGI2
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI1[j]
    W_NGI2[i] = W_NGI1[i] + neighbor_sum

# NGI3
for i, u in enumerate(nodelist):
    neighbor_sum = 0
    for v in G.neighbors(u):
        j = node_index[v]
        neighbor_sum += W_NGI2[j]
    W_NGI3[i] = W_NGI2[i] + neighbor_sum


print("W_NLI1:", W_NLI1)
print("W_NLI2:", W_NLI2)
print("W_NLI3:", W_NLI3)

print("W_NGI1:", W_NGI1)
print("W_NGI2:", W_NGI2)
print("W_NGI3:", W_NGI3)

W_NLI1: [5.69085035e+01 6.27572877e+00 1.67352770e+01 4.18381562e-01
 1.67352625e+00 1.46433672e+01 1.42249855e+01 1.25514578e+01
 2.51028937e+00 2.09190959e+00 1.38066035e+01 1.08779300e+01
 3.34705535e+00 2.21742420e+01 2.84499709e+01 4.18381562e-01
 2.92867347e+00 1.17146939e+01 2.92867348e+01 6.27572887e+00
 3.34705534e+00 2.09190959e+00 3.34705249e+00 6.27572887e+00
 2.76132071e+01 3.93279010e+01 1.42249855e+01 6.69411081e+00
 1.67352627e+00 5.02058302e+00 8.36763124e-01 4.76955395e+01
 2.51028937e+00 5.85734187e+00 9.20440236e+00 2.67764432e+01
 5.10425949e+01 5.52264133e+01 4.18381562e-01 7.94925658e+00
 4.76955395e+01 5.43896503e+00 7.53087464e+00 2.92867343e+00
 3.55624637e+01 2.92867093e+00 8.36763124e-01 1.92455686e+01
 8.36763124e-01 3.76543406e+00 3.76543732e+00 8.36763124e-01
 2.09190781e+00 6.69411081e+00 7.53087464e+00 7.53087464e+00
 1.67352625e+00 1.25514471e+01 8.78602029e+00 4.60220117e+00
 1.25514469e+00 3.13786444e+01 4.56036299e+01 5.02058310e+00
 8.36763124e-01 

### 🔹 Neighborhood Matrix Construction

**Definition:**
A neighborhood matrix is constructed for each node to represent its local structural information using a fixed-size subgraph.

**Core Idea:**
Instead of using the entire graph, a localized neighborhood subgraph is extracted for each node, ensuring:
*   Reduced computational complexity
*   Consistent input size for learning models

---

### ⚙️ Computation Steps:
1.  **Extract** one-hop neighbors of the target node.
2.  **Rank** neighbors based on importance scores (e.g., $W_{NLI3}$ or $W_{NGI3}$).
3.  **Select** the top $L$ neighbors.
4.  **Construct** a $(L+1) 	imes (L+1)$ adjacency matrix including the node and selected neighbors.

**Role of Parameter $L$:**
*   Determines the size of the neighborhood.
*   Controls how much local information is captured.
*   Ensures uniform matrix size across all nodes.

---

### ✅ Key Advantage
*   **Efficiency:** Reduces computational complexity.
*   **Robustness:** Avoids bias from high-degree nodes.
*   **Consistency:** Provides structured and consistent input for GCN.

**Interpretation:**
Each node is represented by a fixed-size local subgraph, capturing its most important neighbors and their mutual connections.

In [12]:
import numpy as np

# choose L <= max neighbors: use a fixed neighborhood size of 40 for the large graph
L = 40

def neighborhood_matrix(node):

    nbrs = list(G.neighbors(node))

    # sort neighbors using importance (W_NLI3 or W_NGI3)
    nbrs_sorted = sorted(nbrs, key=lambda x: W_NLI3[nodelist.index(x)], reverse=True)

    nbrs_selected = nbrs_sorted[:L]

    # keep a fixed size L+1; pad with placeholder values if the node has fewer neighbors
    nodes = [node] + nbrs_selected
    if len(nodes) < L + 1:
        nodes += [None] * (L + 1 - len(nodes))

    size = L + 1
    mat = np.zeros((size, size))

    for i, u in enumerate(nodes):
        for j, v in enumerate(nodes):
            if u is not None and v is not None and G.has_edge(u, v):
                mat[i, j] = 1

    return mat, nodes


# Example: for the first node in the graph
mat0, nodes0 = neighborhood_matrix(nodelist[0])

print("Neighborhood Matrix:\n", mat0)
print("Nodes used:", nodes0)

Neighborhood Matrix:
 [[0. 1. 1. ... 1. 1. 1.]
 [1. 0. 1. ... 1. 1. 1.]
 [1. 1. 0. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 0. 0. 0.]
 [1. 1. 1. ... 0. 0. 1.]
 [1. 1. 1. ... 0. 1. 0.]]
Nodes used: [np.int64(0), np.int64(1), np.int64(2), np.int64(17), np.int64(7), np.int64(6), np.int64(10), np.int64(9), np.int64(16), np.int64(20), np.int64(5), np.int64(4), np.int64(14), np.int64(11), np.int64(8), np.int64(27), np.int64(18), np.int64(21), np.int64(37), np.int64(36), np.int64(32), np.int64(3), np.int64(25), np.int64(19), np.int64(24), np.int64(40), np.int64(23), np.int64(15), np.int64(51), np.int64(13), np.int64(12), np.int64(30), np.int64(28), np.int64(29), np.int64(41), np.int64(22), np.int64(44), np.int64(39), np.int64(26), np.int64(46), np.int64(50)]


### 🔹 Structural Channel Construction

**Definition:**
Structural channel construction embeds node feature information into the neighborhood matrix to generate multiple feature-aware representations of each node.

**Core Idea:**
Instead of using only structural adjacency, node features are incorporated into the matrix to create channels that capture both:
*   **Structural relationships**
*   **Node importance**

---

### ⚙️ Computation:
For each node, a neighborhood matrix is constructed and node features (e.g., NLI, NGI) are embedded into this matrix according to specific rules:

**Channel Construction Rules:**
*   **Diagonal elements:** Represent the feature value of the node itself.
*   **Off-diagonal elements:**
    *   If an edge exists → assign the feature value of the neighbor.
    *   If no edge exists → the value remains zero.

**Channels Created:**
*   **Local influence channels:** $E^{(NLI1)}$, $E^{(NLI2)}$, $E^{(NLI3)}$
*   **Global influence channels:** $E^{(NGI1)}$, $E^{(NGI2)}$, $E^{(NGI3)}$

---

### ✅ Key Advantage
*   **Integration:** Combines structural and feature information seamlessly.
*   **Power:** Enhances the representation power of nodes.
*   **Scalability:** Provides multi-scale learning capability.

**Interpretation:**
Each channel represents a feature-enriched local subgraph, enabling the model to learn both node importance and connectivity patterns simultaneously.

In [13]:
import numpy as np


def embed_channel(mat, nodes, feature_dict):

    size = mat.shape[0]
    out = np.zeros((size, size))

    for i in range(size):
        for j in range(size):

            u = nodes[i]
            v = nodes[j]

            # diagonal → self feature
            if i == j:
                out[i, j] = feature_dict.get(u, 0)

            # edge exists → take neighbor feature
            elif mat[i, j] == 1:
                out[i, j] = feature_dict.get(v, 0)

    return out

### 🔹 Purpose of Structural Channel Construction

Structural channel construction is performed to transform the graph into a format that can effectively capture both **node importance** and **local structural relationships** in a unified representation.

Graph data is inherently irregular, where each node may have a different number of neighbors. This makes it difficult to directly apply deep learning models that require fixed-size inputs.

To address this, a neighborhood matrix is first constructed for each node, ensuring a consistent structure. However, this matrix only represents connectivity and does not include any information about node importance.

Therefore, node features such as local influence (NLI) and global influence (NGI) are embedded into the neighborhood matrix to create **feature-aware channels**.

**In these channels:**
*   The **diagonal elements** represent the importance of the node itself
*   The **off-diagonal elements** represent the importance of neighboring nodes if a connection exists

This transformation allows the model to simultaneously learn:
1.  **Who is connected to whom** (structure)
2.  **How important each node is** (features)

By constructing multiple channels at different scales (NLI1–3 and NGI1–3), the model is able to capture multi-level influence propagation, improving its ability to identify key nodes.

---

### ✅ Key Benefit
This approach enables the graph to be represented as a **multi-channel matrix** (similar to images), making it suitable for deep learning models while preserving both structural and semantic information.

In [14]:
# convert arrays to dict (important)
NLI_dict = {node: NLI[i] for i, node in enumerate(nodelist)}
NGI_dict = {node: NGI[i] for i, node in enumerate(nodelist)}

# example for node 3
mat, nodes = neighborhood_matrix(3)

E_NLI1 = embed_channel(mat, nodes, NLI_dict)
E_NLI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI2)))
E_NLI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NLI3)))

E_NGI1 = embed_channel(mat, nodes, NGI_dict)
E_NGI2 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI2)))
E_NGI3 = embed_channel(mat, nodes, dict(zip(nodelist, W_NGI3)))

print("E_NLI1:\n", E_NLI1)
print("E_NLI2:\n", E_NLI2)
print("E_NLI3:\n", E_NLI3)
print("E_NGI1:\n", E_NGI1)
print("E_NGI2:\n", E_NGI2)
print("E_NGI3:\n", E_NGI3)

E_NLI1:
 [[31.79702632 56.90850352 56.89989342 ... 14.22498546 12.13307583
  12.55145776]
 [31.79702632 56.90850352 56.89989342 ... 14.22498546 12.13307583
  12.55145776]
 [31.79702632 56.90850352 56.89989342 ... 14.22498546 12.13307583
  12.55145776]
 ...
 [31.79702632 56.90850352 56.89989342 ... 14.22498546  0.
  12.55145776]
 [31.79702632 56.90850352 56.89989342 ...  0.         12.13307583
  12.55145776]
 [31.79702632 56.90850352 56.89989342 ... 14.22498546 12.13307583
  12.55145776]]
E_NLI2:
 [[1473.10717822 1923.31010544 1889.50135294 ...  905.77794075
   909.20140449  883.15181791]
 [1473.10717822 1923.31010544 1889.50135294 ...  905.77794075
   909.20140449  883.15181791]
 [1473.10717822 1923.31010544 1889.50135294 ...  905.77794075
   909.20140449  883.15181791]
 ...
 [1473.10717822 1923.31010544 1889.50135294 ...  905.77794075
     0.          883.15181791]
 [1473.10717822 1923.31010544 1889.50135294 ...    0.
   909.20140449  883.15181791]
 [1473.10717822 1923.31010544 1889.5

### 🔹 Channel Tensor Construction

**Definition:**
Channel tensor construction combines multiple feature-embedded neighborhood matrices into a unified multi-dimensional representation for each node.

**Core Idea:**
For each node, six structural channels are generated by embedding different feature representations ($NLI_1$–$NLI_3$ and $NGI_1$–$NGI_3$) into the neighborhood matrix.

---

### ⚙️ Computation:
1.  A **neighborhood matrix** of size $(L+1) \times (L+1)$ is constructed.
2.  **Six feature matrices** are generated by embedding node importance values into the structural layout.
3.  These matrices are stacked to form a tensor of size:
    $$6 \times (L+1) \times (L+1)$$

---

### ✅ Key Advantage
*   **Multi-Perspective:** Captures multiple levels of node importance.
*   **Hybrid Representation:** Combines structural connectivity and feature importance.
*   **Deep Learning Ready:** Enables standard CNN or GCN models to process graph data efficiently.

**Interpretation:**
Each node is represented as a multi-channel tensor, where each channel encodes a different aspect of node influence and neighborhood structure.

In [15]:
channels = []

for node in G.nodes():

    mat, nodes = neighborhood_matrix(node)

    # create feature dicts (node → value), skipping None placeholders
    f1 = {n: NLI_dict.get(n, 0) for n in nodes}
    f2 = {n: W_NLI2[node_index[n]] if n is not None else 0 for n in nodes}
    f3 = {n: W_NLI3[node_index[n]] if n is not None else 0 for n in nodes}

    f4 = {n: NGI_dict.get(n, 0) for n in nodes}
    f5 = {n: W_NGI2[node_index[n]] if n is not None else 0 for n in nodes}
    f6 = {n: W_NGI3[node_index[n]] if n is not None else 0 for n in nodes}

    # create channels
    c1 = embed_channel(mat, nodes, f1)
    c2 = embed_channel(mat, nodes, f2)
    c3 = embed_channel(mat, nodes, f3)

    c4 = embed_channel(mat, nodes, f4)
    c5 = embed_channel(mat, nodes, f5)
    c6 = embed_channel(mat, nodes, f6)

    # stack → (6, L+1, L+1)
    tensor = np.stack([c1, c2, c3, c4, c5, c6])

    channels.append(tensor)

# final shape: (num_nodes, 6, L+1, L+1)
channels = np.array(channels)

print(channels.shape)

(500, 6, 41, 41)


### 🔹 Channel Attention Module

**Definition:**
The channel attention module is used to adaptively learn the importance of different feature channels and enhance the representation of informative channels.

**Core Idea:**
Not all feature channels contribute equally to node importance. Therefore, an attention mechanism is introduced to assign weights to each channel dynamically.

---

### ⚙️ Computation:
1.  **Global average pooling** is applied to each channel to obtain a compact representation.
2.  The pooled values are passed through **fully connected layers**.
3.  A **sigmoid activation** generates normalized weights between 0 and 1.
4.  These weights are **multiplied** with the input feature maps.

---

### ✅ Key Advantage
*   **Feature Selection:** Highlights important feature channels.
*   **Noise Reduction:** Suppresses less relevant information.
*   **Robustness:** Improves model robustness and generalization.

**Interpretation:**
Channels representing more meaningful structural or influence patterns receive higher weights, allowing the model to focus on the most relevant information.

In [16]:
import torch
import torch.nn as nn

class ChannelAttention(nn.Module):

    def __init__(self, channels=6, reduction=2):
        super(ChannelAttention, self).__init__()

        # Global Average Pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        # Fully Connected Layers (SE block)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x: (batch, channels, height, width)

        b, c, _, _ = x.size()

        # Step 1: Global Average Pooling
        y = self.avg_pool(x).view(b, c)

        # Step 2: FC → channel weights
        y = self.fc(y).view(b, c, 1, 1)

        # Step 3: Multiply weights
        out = x * y

        return out

In [17]:
# test input
x = torch.randn(2, 6, 3, 3)   # batch=2, channels=6

model = ChannelAttention(6)

out = model(x)

print("Input shape:", x.shape)
print("Output shape:", out.shape)

Input shape: torch.Size([2, 6, 3, 3])
Output shape: torch.Size([2, 6, 3, 3])


### 🔹 NLGCN Model Architecture

**Definition:**
The NLGCN model is a convolutional neural network designed to learn node influence from multi-channel structural representations of graph data.

**Core Idea:**
The model processes the constructed channel tensor using convolutional layers to extract structural patterns, while a channel attention mechanism enhances important feature channels.

---

### 🏗 Architecture:
*   **Input:** Multi-channel tensor of size $6 \times (L+1) \times (L+1)$.
*   **Channel Attention:** Assigns adaptive weights to feature channels.
*   **Convolution Layer 1:** Extracts local structural patterns (followed by Batch Normalization and ReLU).
*   **Pooling Layer:** Reduces spatial dimensions and retains key features.
*   **Convolution Layer 2:** Learns higher-level structural representations.
*   **Fully Connected Layers:** Transform extracted features into the final influence score.

---

### ✅ Key Advantage
*   **Hybrid Learning:** Captures both local and multi-scale structural patterns.
*   **Attention-Driven:** Enhances feature learning using the attention mechanism.
*   **Structured Processing:** Efficiently processes graph data in a consistent matrix format.

**Interpretation:**
The model learns how node importance is influenced by both its local structure and multi-scale neighborhood features, producing a final influence score.

### 🛠 Model Component Summary

| Part | Role |
| :--- | :--- |
| **Channel Attention** | Adaptively select and weight important feature channels |
| **Convolution Layer 1** | Extract local structural patterns from the neighborhood |
| **Max Pooling** | Reduce spatial dimensions and retain significant features |
| **Convolution Layer 2** | Learn higher-order structural representations |
| **Fully Connected** | Map structural features to the final node influence score |

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class NLGCN(nn.Module):

    def __init__(self):
        super(NLGCN, self).__init__()

        self.attention = ChannelAttention(6)

        # Conv 1
        self.conv1 = nn.Conv2d(6, 16, kernel_size=2)
        self.bn = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment this block when using larger graphs (L >= 4 or bigger input size)
        # self.conv2 = nn.Conv2d(16, 32, kernel_size=2)
        # self.pool2 = nn.MaxPool2d(2)

        # -------- FC Layers --------
        # For L=40, input size after conv1+pool is 16 x 20 x 20
        self.fc1 = nn.Linear(16 * 20 * 20, 8)
        self.fc2 = nn.Linear(8, 1)

        # For even larger graphs or extra conv layers, adjust this accordingly
        # self.fc1 = nn.Linear(32 * k * k, 64)  # adjust k based on output size
        # self.fc2 = nn.Linear(64, 1)

    def forward(self, x):

        # x: (batch, 6, L+1, L+1)

        x = self.attention(x)

        x = self.conv1(x)
        x = self.bn(x)
        x = F.relu(x)

        x = self.pool(x)

        # -------- Conv 2 (for larger graphs) --------
        # Uncomment when input size is large enough
        # x = self.conv2(x)
        # x = F.relu(x)
        # x = self.pool2(x)

        x = x.view(x.size(0), -1)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)

        return x

### 🧬 SIR-Based Label Generation

**Definition:**
The SIR (Susceptible–Infected–Recovered) model is used to generate ground truth labels representing node influence.

---

### ⚙️ Computation:

1.  **Epidemic Threshold:** The threshold is calculated as:
    $$\beta_c = \frac{\langle k \rangle}{\langle k^2 \rangle - \langle k \rangle}$$

2.  **Infection Probability:** The probability is set relative to the threshold:
    $$\beta = 1.5\beta_c$$

3.  **Simulation Process:**
    *   Each node is treated as the initial infected node.
    *   The SIR process is simulated multiple times (e.g., 500 runs).
    *   The average number of recovered nodes is calculated as the influence score.

---

### ✅ Normalization:
The labels are normalized to the range $[0, 1]$ to ensure stable model training.

**Interpretation:**
Nodes that infect a larger portion of the network in the SIR simulation are considered more influential and receive higher ground truth scores.

In [19]:
import numpy as np
import random

# ---- Degree calculations ----
deg = np.array([d for n, d in G.degree()])

k_avg = np.mean(deg)
k2_avg = np.mean(deg**2)

beta_c = k_avg / (k2_avg - k_avg)

beta = 1.5 * beta_c
mu = 1


# ---- SIR Simulation ----
def SIR_simulation(G, seed, beta, mu, steps=1000):

    susceptible = set(G.nodes())
    infected = {seed}
    recovered = set()

    susceptible.remove(seed)

    for _ in range(steps):

        new_infected = set()
        new_recovered = set()

        for node in infected:

            # spread infection
            for nbr in G.neighbors(node):
                if nbr in susceptible:
                    if random.random() < beta:
                        new_infected.add(nbr)

            # recovery
            if random.random() < mu:
                new_recovered.add(node)

        infected |= new_infected
        infected -= new_recovered

        recovered |= new_recovered
        susceptible -= new_infected

        if len(infected) == 0:
            break

    return len(recovered)


# ---- Label Generation ----
labels = []
runs = 500

for node in G.nodes():

    spread = 0

    for _ in range(runs):
        spread += SIR_simulation(G, node, beta, mu)

    labels.append(spread / runs)

labels = np.array(labels)


# ---- Normalize Labels ----
labels = labels / np.max(labels)

print("Labels:", labels)

Labels: [0.99479228 0.31098355 0.55260978 0.06338028 0.08734762 0.50100604
 0.36454018 0.51651083 0.17386673 0.15161558 0.65812522 0.51313765
 0.19315895 0.72310333 0.80435555 0.0575216  0.19795242 0.46839863
 0.81181205 0.29766836 0.1919162  0.16333294 0.21558764 0.30370458
 0.81690141 0.87927565 0.61675938 0.35294118 0.14273879 0.28103918
 0.07823411 0.95887087 0.13409871 0.19931353 0.4236596  0.77305007
 0.89430702 1.         0.06456385 0.42632264 0.93443011 0.30210676
 0.3418156  0.22310333 0.85814889 0.11735117 0.08000947 0.71789561
 0.08166647 0.18487395 0.30701858 0.10261569 0.11634513 0.38832998
 0.38785655 0.41064031 0.08154811 0.4576873  0.36294236 0.29530122
 0.10468695 0.8680909  0.89679252 0.23032312 0.08859037 0.66954669
 0.3155403  0.16309622 0.23831223 0.568588   0.71446325 0.80145579
 0.83950763 0.33305717 0.76293052 0.39442538 0.10930288 0.11273523
 0.77689667 0.05633803 0.81666469 0.32181323 0.78778554 0.07592615
 0.75707184 0.12255888 0.38637709 0.69818913 0.6816191

In [20]:
import torch
import torch.nn as nn

# ---- Convert data to tensors ----
X = torch.tensor(channels, dtype=torch.float32)

# ---- Normalize input channels per channel ----
X = X - X.mean(dim=(0, 2, 3), keepdim=True)
X = X / (X.std(dim=(0, 2, 3), keepdim=True) + 1e-6)

# ---- Normalize labels for stable regression training ----
y = torch.tensor(labels, dtype=torch.float32).view(-1, 1)
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / (y_std + 1e-6)

print("X mean per channel:", X.mean(dim=(0, 2, 3)))
print("X std per channel:", X.std(dim=(0, 2, 3)))
print("y mean:", y_mean.item(), "y std:", y_std.item())

# ---- Initialize model ----
model = NLGCN()

# ---- Optimizer ----
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ---- Loss function ----
criterion = nn.MSELoss()

# ---- Training Loop ----
epochs = 300

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X)
    loss = criterion(outputs, y)
    loss.backward()
    optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss = {loss.item():.6f}")

# ---- Final predictions ----
model.eval()
with torch.no_grad():
    predictions = model(X)

print("\nFinal Predictions:\n", predictions)

X mean per channel: tensor([-1.6049e-08, -5.3156e-08,  2.8364e-07,  1.2309e-07,  3.7928e-07,
         1.2403e-07])
X std per channel: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])
y mean: 0.1796468198299408 y std: 0.22189712524414062
Epoch 0, Loss = 1.531484
Epoch 20, Loss = 0.026413
Epoch 40, Loss = 0.008665
Epoch 60, Loss = 0.006489
Epoch 80, Loss = 0.005264
Epoch 100, Loss = 0.004600
Epoch 120, Loss = 0.004062
Epoch 140, Loss = 0.003611
Epoch 160, Loss = 0.003200
Epoch 180, Loss = 0.002842
Epoch 200, Loss = 0.002539
Epoch 220, Loss = 0.002323
Epoch 240, Loss = 0.002170
Epoch 260, Loss = 0.002048
Epoch 280, Loss = 0.001948

Final Predictions:
 tensor([[ 3.6680e+00],
        [ 5.4056e-01],
        [ 1.6848e+00],
        [-5.3141e-01],
        [-3.4531e-01],
        [ 1.4511e+00],
        [ 8.4014e-01],
        [ 1.5208e+00],
        [-7.9251e-02],
        [-1.1545e-01],
        [ 2.1636e+00],
        [ 1.5147e+00],
        [ 1.7665e-02],
        [ 2.4638e+00],
        [ 2.8

### 🔹 Prediction and Ranking Evaluation

**Definition:**
After training, the model predicts influence scores for each node, which are used to rank nodes based on their importance.

---

### ⚙️ Computation:
1.  **Generate Predicted Scores:** The trained model is used to compute influence scores for all nodes in the graph.
2.  **Predicted Ranking:** Nodes are ranked in descending order based on these predicted scores.
3.  **Ground Truth Ranking:** A reference ranking is obtained from the SIR-based simulation labels.
4.  **Comparison:** The predicted ranking is compared with the SIR ranking to measure alignment.

---

### 📊 Evaluation:
*   **Top-k Comparison:** The top-ranked nodes from both predicted and ground truth sets are compared to assess how well the model identifies the most influential nodes.
*   **Ranking Correlation:** Statistical measures can be used to determine the accuracy of the overall node order.

**Key Insight:**
The closer the predicted ranking is to the SIR ranking, the better the model captures the underlying dynamics of node influence within the network.

In [21]:
import numpy as np
import torch

# ---- Prediction ----
model.eval()

with torch.no_grad():
    pred = model(X).detach().cpu().numpy().flatten()

print("Predicted scores:", pred)


# ---- Ranking ----
ranking_pred = np.argsort(pred)[::-1]
ranking_true = np.argsort(labels)[::-1]

print("\nTop predicted nodes:", ranking_pred)
print("Top SIR nodes:", ranking_true)

# Top-k comparison
k = 10
print(f"\nTop {k} predicted nodes:", ranking_pred[:k])
print(f"Top {k} SIR nodes:", ranking_true[:k])

Predicted scores: [ 3.66798663e+00  5.40560126e-01  1.68478119e+00 -5.31411767e-01
 -3.45310837e-01  1.45106697e+00  8.40144515e-01  1.52082384e+00
 -7.92512000e-02 -1.15447268e-01  2.16358614e+00  1.51469254e+00
  1.76652968e-02  2.46384597e+00  2.81865978e+00 -5.31411767e-01
  1.14670128e-01  1.30815256e+00  2.85263562e+00  5.99644065e-01
  8.87091160e-02 -1.14501446e-01  1.18001908e-01  5.91267228e-01
  2.89205003e+00  3.13840961e+00  1.97202325e+00  7.98477530e-01
 -2.17677563e-01  4.26194280e-01 -4.11117435e-01  3.61316156e+00
 -1.29804671e-01  1.18149489e-01  1.09228265e+00  2.68299961e+00
  3.20429158e+00  3.61084962e+00 -5.31411767e-01  1.11698568e+00
  3.41053939e+00  5.28515697e-01  7.35054731e-01  1.06050640e-01
  3.06418586e+00 -1.31116748e-01 -4.11117435e-01  2.43752408e+00
 -4.11117435e-01  9.06707346e-02  3.49275619e-01 -4.11117435e-01
 -1.53927594e-01  9.32014823e-01  9.50774431e-01  1.03537643e+00
 -4.39127624e-01  1.25532854e+00  8.24964166e-01  4.40910727e-01
 -3.023

###  Model Evaluation

**Kendall Tau Correlation:**
The Kendall Tau coefficient is used to measure the similarity between the predicted node ranking and the SIR-based ground truth ranking. A higher value indicates better agreement between the two rankings.

**Top-N Influence Spread:**
The top-N nodes predicted by the model are selected, and their spreading capability is evaluated using the SIR model. The total number of infected nodes represents the effectiveness of the selected nodes.

---

### ✅ Key Insight:
*   **Kendall Tau:** Evaluates the overall ranking consistency.
*   **Top-N Spread:** Evaluates the practical influence performance of the predicted top nodes.

In [22]:
from scipy.stats import kendalltau

# ---- Kendall Tau Correlation ----
tau, p = kendalltau(pred, labels)

print("Kendall Tau:", tau)


# ---- Top-N Influence Spread ----
N = 3   # for small graph (you can change)

top_node_indices = ranking_pred[:N]
top_nodes = [nodelist[idx] for idx in top_node_indices]

spread_total = 0

for node in top_nodes:
    spread_total += SIR_simulation(G, node, beta, mu)

print("Top-N node indices:", top_node_indices)
print("Top-N nodes:", top_nodes)
print("Spread ability:", spread_total)

Kendall Tau: 0.8947595052319491
Top-N node indices: [100   0  31]
Top-N nodes: [np.int64(1), np.int64(0), np.int64(17)]
Spread ability: 111


In [23]:
import networkx as nx
from scipy.stats import kendalltau

# Create a copy of G without self-loops for traditional centrality calculations
G_clean = G.copy()
G_clean.remove_edges_from(nx.selfloop_edges(G_clean))

print('Calculating traditional centrality measures for US airports...')

# Degree Centrality
deg_dict = nx.degree_centrality(G_clean)
deg_cent = np.array([deg_dict[n] for n in nodelist])
tau_deg, _ = kendalltau(pred, deg_cent)
print(f'Kendall Tau (Prediction vs Degree): {tau_deg:.4f}')

# Betweenness Centrality
bet_dict = nx.betweenness_centrality(G_clean)
bet_cent = np.array([bet_dict[n] for n in nodelist])
tau_bet, _ = kendalltau(pred, bet_cent)
print(f'Kendall Tau (Prediction vs Betweenness): {tau_bet:.4f}')

# Closeness Centrality
clos_dict = nx.closeness_centrality(G_clean)
clos_cent = np.array([clos_dict[n] for n in nodelist])
tau_clos, _ = kendalltau(pred, clos_cent)
print(f'Kendall Tau (Prediction vs Closeness): {tau_clos:.4f}')

# PageRank
pr_dict = nx.pagerank(G_clean)
pr_cent = np.array([pr_dict[n] for n in nodelist])
tau_pr, _ = kendalltau(pred, pr_cent)
print(f'Kendall Tau (Prediction vs PageRank): {tau_pr:.4f}')

# Coreness (k-core)
core_dict = nx.core_number(G_clean)
core_cent = np.array([core_dict[n] for n in nodelist])
tau_core, _ = kendalltau(pred, core_cent)
print(f'Kendall Tau (Prediction vs Coreness): {tau_core:.4f}')

# Eigenvector Centrality
try:
    eig_dict = nx.eigenvector_centrality(G_clean, max_iter=1000)
    eig_cent = np.array([eig_dict[n] for n in nodelist])
    tau_eig, _ = kendalltau(pred, eig_cent)
    print(f'Kendall Tau (Prediction vs Eigenvector): {tau_eig:.4f}')
except Exception as e:
    print(f'Eigenvector centrality failed: {e}')


Calculating traditional centrality measures for US airports...
Kendall Tau (Prediction vs Degree): 0.6645
Kendall Tau (Prediction vs Betweenness): 0.3578
Kendall Tau (Prediction vs Closeness): 0.8151
Kendall Tau (Prediction vs PageRank): 0.3861
Kendall Tau (Prediction vs Coreness): 0.7071
Kendall Tau (Prediction vs Eigenvector): 0.9307


In [24]:
# ============================================================
# Weighted Centrality Correlation Analysis
# Weight Semantics: HIGH weight = ENEMIES (adversarial/costly)
# So weight is inversely proportional to influence strength.
# We convert: effective_weight = 1 / raw_weight
# This means a high-weight (enemy) edge contributes LESS
# to centrality — reflecting reduced influence flow.
# ============================================================

import numpy as np
import networkx as nx
from scipy.stats import kendalltau

# ---- Step 1: Reload the graph WITH weights ----
# Budapest.txt format: V1  V2  weight (tab/space separated)
path = os.path.join("..", "Datasets",  "US_airports.txt")
raw = np.loadtxt(path)

if raw.ndim == 1:
    raw = raw.reshape(1, -1)

# Build a weighted graph
G_weighted = nx.Graph()
for row in raw:
    u, v, w = int(row[0]), int(row[1]), float(row[2])

    # Safety: avoid zero or negative weights before inversion
    w = abs(w) if w != 0 else 1e-6

    # Inverse weight: high raw weight → low influence (enemy logic)
    inv_w = 1.0 / w

    # If edge already exists keep the minimum inv_w (strongest enemy = weakest link)
    if G_weighted.has_edge(u, v):
        existing = G_weighted[u][v]['weight']
        G_weighted[u][v]['weight'] = min(existing, inv_w)
    else:
        G_weighted.add_edge(u, v, weight=inv_w)

# Remove self-loops (same as unweighted pipeline)
G_weighted.remove_edges_from(nx.selfloop_edges(G_weighted))

print(f"Weighted graph — Nodes: {G_weighted.number_of_nodes()}, Edges: {G_weighted.number_of_edges()}")
print(f"Sample inverse weights: {[round(G_weighted[u][v]['weight'], 4) for u,v in list(G_weighted.edges())[:5]]}")

# ---- Step 2: Compute Weighted Centrality Measures ----
print("\nCalculating weighted centrality measures for Budapest...")

# -- Weighted Degree (Strength) --
# Sum of inv_weights on edges — low-weight enemies reduce strength
strength_dict = dict(G_weighted.degree(weight='weight'))
strength_cent  = np.array([strength_dict.get(n, 0.0) for n in nodelist])
# Normalize to [0,1] for fair comparison
strength_cent  = strength_cent / strength_cent.max() if strength_cent.max() > 0 else strength_cent
tau_wdeg, _ = kendalltau(pred, strength_cent)
print(f"Kendall Tau (Prediction vs Weighted Degree / Strength): {tau_wdeg:.4f}")

# -- Weighted Betweenness --
# Uses inv_weight as distance — high-weight (enemy) edges are LONGER paths
# so they are avoided by shortest paths, reducing betweenness of bridge nodes
wbet_dict  = nx.betweenness_centrality(G_weighted, weight='weight', normalized=True)
wbet_cent  = np.array([wbet_dict[n] for n in nodelist])
tau_wbet, _ = kendalltau(pred, wbet_cent)
print(f"Kendall Tau (Prediction vs Weighted Betweenness):       {tau_wbet:.4f}")

# -- Weighted Closeness --
# distance = inv_weight → enemy edges make nodes "farther apart"
wclos_dict = nx.closeness_centrality(G_weighted, distance='weight')
wclos_cent = np.array([wclos_dict[n] for n in nodelist])
tau_wclos, _ = kendalltau(pred, wclos_cent)
print(f"Kendall Tau (Prediction vs Weighted Closeness):         {tau_wclos:.4f}")

# -- Weighted PageRank --
# edge weight = transition probability proxy (inv_w = low for enemy edges)
# so enemy edges pass less rank to neighbors
wpr_dict   = nx.pagerank(G_weighted, weight='weight')
wpr_cent   = np.array([wpr_dict[n] for n in nodelist])
tau_wpr, _ = kendalltau(pred, wpr_cent)
print(f"Kendall Tau (Prediction vs Weighted PageRank):          {tau_wpr:.4f}")

# -- Weighted Eigenvector Centrality --
# Propagates score proportional to inv_weight of connecting edges
# Enemy edges (high raw w → low inv_w) reduce neighbor's contribution
try:
    weig_dict  = nx.eigenvector_centrality(G_weighted, weight='weight', max_iter=1000)
    weig_cent  = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")
except nx.PowerIterationFailedConvergence:
    print("Weighted Eigenvector did not converge — trying numpy fallback...")
    weig_dict  = nx.eigenvector_centrality_numpy(G_weighted, weight='weight')
    weig_cent  = np.array([weig_dict[n] for n in nodelist])
    tau_weig, _ = kendalltau(pred, weig_cent)
    print(f"Kendall Tau (Prediction vs Weighted Eigenvector):       {tau_weig:.4f}")

# ---- Step 3: Side-by-side comparison table ----
print("\n" + "="*62)
print(f"{'Measure (USairports)':<30} {'Unweighted':>12} {'Weighted':>12}")
print("="*62)
print(f"{'Degree / Strength':<30} {tau_deg:>12.4f} {tau_wdeg:>12.4f}")
print(f"{'Betweenness':<30} {tau_bet:>12.4f} {tau_wbet:>12.4f}")
print(f"{'Closeness':<30} {tau_clos:>12.4f} {tau_wclos:>12.4f}")
print(f"{'PageRank':<30} {tau_pr:>12.4f} {tau_wpr:>12.4f}")
print(f"{'Eigenvector':<30} {tau_eig:>12.4f} {tau_weig:>12.4f}")
print("="*62)
print("Weight semantics: high raw weight = adversarial edge")
print("Effective weight = 1 / raw_weight (enemy edges penalized)")

Weighted graph — Nodes: 500, Edges: 2980
Sample inverse weights: [0.0, 0.0, 0.0, 0.0, 0.0]

Calculating weighted centrality measures for Budapest...
Kendall Tau (Prediction vs Weighted Degree / Strength): 0.3248
Kendall Tau (Prediction vs Weighted Betweenness):       0.1128
Kendall Tau (Prediction vs Weighted Closeness):         0.6832
Kendall Tau (Prediction vs Weighted PageRank):          0.2389
Kendall Tau (Prediction vs Weighted Eigenvector):       0.5899

Measure (USairports)             Unweighted     Weighted
Degree / Strength                    0.6645       0.3248
Betweenness                          0.3578       0.1128
Closeness                            0.8151       0.6832
PageRank                             0.3861       0.2389
Eigenvector                          0.9307       0.5899
Weight semantics: high raw weight = adversarial edge
Effective weight = 1 / raw_weight (enemy edges penalized)
